# CHIVES Data — Column Rename for Dashboard Upload

Renames columns in the CHIVES (Chicago Healthy and Equitable Index by Space) dataset
to match the **socioeconomics** domain expected by the dashboard, and adds community
area centroid coordinates (required for map rendering).

In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\shanm\Downloads\chives-data-public.csv", low_memory=False)
print(f"{len(df):,} rows × {len(df.columns)} columns")
df.head(3)

801 rows × 56 columns


,geoid,CEJI,treeChng,treeCCov17,treeCCov10,HeathDeath_Percent,leadPoisonR,ndvi,trees_n,trees_crown_den,...,dsplcPresr,chwAirTemp,chwHeatAve,chwHeatMax,eclipsPM25,lungCRt17,physRt17,adAsthma17,cancerRt17,hyptRt17
0,17031291600,49.68,-0.227,11.56,11.790699,3.27,2.457265,0.192543,453,11.8,...,"Vulnerable, Prices Rising",84.291567,88.925354,99.392030,11.189418,91.371638,25.2,10.0,616.154834,29.7
1,17031580200,94.82,0.152,13.51,13.359938,0.61,1.559934,0.226763,1588,12.8,...,"Vulnerable, Prices Not Rising",84.618429,89.329550,100.217085,11.015970,44.322032,35.8,11.5,366.897263,15.8
2,17031590500,52.21,-2.953,23.67,26.621697,1.23,0.319489,0.405935,2093,27.8,...,"Vulnerable, Prices Rising",84.049685,89.217986,100.277585,10.578465,46.114989,24.0,NaN,439.724737,32.7


In [2]:
# Community-area centroid lookup (community area number → lat, lon)
# Approximate geographic centres for Chicago's 77 community areas.
CENTROIDS = {
    1:  (41.9987, -87.6626),  # Rogers Park
    2:  (41.9958, -87.6888),  # West Ridge
    3:  (41.9655, -87.6542),  # Uptown
    4:  (41.9710, -87.6855),  # Lincoln Square
    5:  (41.9564, -87.6680),  # North Center
    6:  (41.9432, -87.6464),  # Lake View
    7:  (41.9218, -87.6494),  # Lincoln Park
    8:  (41.9001, -87.6343),  # Near North Side
    9:  (41.9966, -87.8123),  # Edison Park
    10: (41.9978, -87.8079),  # Norwood Park
    11: (41.9706, -87.7907),  # Jefferson Park
    12: (41.9985, -87.7614),  # Forest Glen
    13: (41.9815, -87.7217),  # North Park
    14: (41.9682, -87.7218),  # Albany Park
    15: (41.9599, -87.7654),  # Portage Park
    16: (41.9533, -87.7257),  # Irving Park
    17: (41.9482, -87.7964),  # Dunning
    18: (41.9261, -87.7991),  # Montclare
    19: (41.9200, -87.7599),  # Belmont Cragin
    20: (41.9056, -87.7291),  # Hermosa
    21: (41.9339, -87.7161),  # Avondale
    22: (41.9246, -87.7010),  # Logan Square
    23: (41.9012, -87.7178),  # Humboldt Park
    24: (41.8960, -87.6739),  # West Town
    25: (41.8958, -87.7686),  # Austin
    26: (41.8826, -87.7395),  # West Garfield Park
    27: (41.8826, -87.7187),  # East Garfield Park
    28: (41.8740, -87.6668),  # Near West Side
    29: (41.8640, -87.7215),  # North Lawndale
    30: (41.8379, -87.7090),  # South Lawndale
    31: (41.8500, -87.6686),  # Lower West Side
    32: (41.8826, -87.6281),  # Loop
    33: (41.8623, -87.6275),  # Near South Side
    34: (41.8565, -87.6313),  # Armour Square
    35: (41.8424, -87.6195),  # Douglas
    36: (41.8286, -87.6047),  # Oakland
    37: (41.8207, -87.6381),  # Fuller Park
    38: (41.8163, -87.6123),  # Grand Boulevard
    39: (41.8205, -87.5932),  # Kenwood
    40: (41.8060, -87.6234),  # Washington Park
    41: (41.7985, -87.5913),  # Hyde Park
    42: (41.7809, -87.5966),  # Woodlawn
    43: (41.7616, -87.5755),  # South Shore
    44: (41.7514, -87.6200),  # Chatham
    45: (41.7455, -87.5897),  # Avalon Park
    46: (41.7376, -87.5615),  # South Chicago
    47: (41.7203, -87.6037),  # Burnside
    48: (41.7283, -87.5908),  # Calumet Heights
    49: (41.6870, -87.6254),  # Roseland
    50: (41.7052, -87.6089),  # Pullman
    51: (41.7073, -87.5586),  # South Deering
    52: (41.7226, -87.5340),  # East Side
    53: (41.6771, -87.6378),  # West Pullman
    54: (41.6406, -87.6351),  # Riverdale
    55: (41.6462, -87.5508),  # Hegewisch
    56: (41.7876, -87.7814),  # Garfield Ridge
    57: (41.8085, -87.7402),  # Archer Heights
    58: (41.8247, -87.7052),  # Brighton Park
    59: (41.8315, -87.6924),  # McKinley Park
    60: (41.8417, -87.6582),  # Bridgeport
    61: (41.8132, -87.6544),  # New City
    62: (41.7902, -87.7244),  # West Elsdon
    63: (41.7868, -87.7040),  # Gage Park
    64: (41.7800, -87.7685),  # Clearing
    65: (41.7730, -87.7081),  # West Lawn
    66: (41.7638, -87.6946),  # Chicago Lawn
    67: (41.7756, -87.6653),  # West Englewood
    68: (41.7808, -87.6432),  # Englewood
    69: (41.7666, -87.6218),  # Greater Grand Crossing
    70: (41.7456, -87.7218),  # Ashburn
    71: (41.7493, -87.6525),  # Auburn Gresham
    72: (41.7224, -87.6747),  # Beverly
    73: (41.7031, -87.6508),  # Washington Heights
    74: (41.6978, -87.7003),  # Mount Greenwood
    75: (41.6903, -87.6716),  # Morgan Park
    76: (41.9861, -87.9063),  # O'Hare
    77: (41.9788, -87.6601),  # Edgewater
}

In [3]:
# Rename columns to match dashboard socioeconomics domain
rename_map = {
    # === Domain-matched columns (socioeconomics) ===
    "comAreaNm":          "Community Area",
    "hrdshpIndx":         "HARDSHIP INDEX",
    "pop18":              "Population",
    "whiteP18":           "White",
    "blackP18":           "Black",
    "hispP18":            "Hispanic",
    "asianP18":           "Asian",
    "mdRent18":           "MEDIAN_RENT",
    "mdHome18":           "MEDIAN_HOME_VALUE",
    "frgnBrn18":          "FOREIGN_BORN",
    "ovrcwdHs18":         "PERCENT OF HOUSING CROWDED",
    "college18":          "BACHELORS_DEGREE",
    # === Remaining columns — human-readable names ===
    "geoid":              "GEOID",
    "CEJI":               "CEJI_SCORE",
    "treeChng":           "TREE_CANOPY_CHANGE",
    "treeCCov17":         "TREE_CANOPY_2017",
    "treeCCov10":         "TREE_CANOPY_2010",
    "HeathDeath_Percent": "HEALTH_DEATH_PERCENT",
    "leadPoisonR":        "LEAD_POISON_RATE",
    "ndvi":               "NDVI",
    "trees_n":            "TREE_COUNT",
    "trees_crown_den":    "TREE_CROWN_DENSITY",
    "socvlnIndx":         "SOCIAL_VULNERABILITY_INDEX",
    "urbanFlood":         "URBAN_FLOOD_RISK",
    "surfcTemp":          "SURFACE_TEMP",
    "trafficVol":         "TRAFFIC_VOLUME",
    "nnetPM25":           "PM25",
    "chAsthmaED":         "CHILD_ASTHMA_ED_RATE",
    "chAsthma17":         "CHILD_ASTHMA_RATE",
    "amIndP18":           "NATIVE_AMERICAN",
    "other18":            "OTHER_RACE",
    "chldren18":          "CHILDREN_PERCENT",
    "senior18":           "SENIOR_PERCENT",
    "zipCode":            "ZIP_CODE",
    "comArea":            "COMMUNITY_AREA_NUMBER",
    "plantDivrs":         "PLANT_DIVERSITY",
    "plantTotl":          "PLANT_TOTAL",
    "plantSpec":          "PLANT_SPECIES",
    "redLined":           "RED_LINED",
    "hhldSize18":         "HOUSEHOLD_SIZE",
    "fmHhld18":           "FAMILY_HOUSEHOLD_PERCENT",
    "popDens18":          "POPULATION_DENSITY",
    "costBurd18":         "COST_BURDEN",
    "hsngSuscp":          "HOUSING_SUSCEPTIBILITY",
    "hsngVuln":           "HOUSING_VULNERABILITY",
    "hsngChng":           "HOUSING_CHANGE",
    "dsplcPresr":         "DISPLACEMENT_PRESSURE",
    "chwAirTemp":         "AIR_TEMP",
    "chwHeatAve":         "HEAT_AVE",
    "chwHeatMax":         "HEAT_MAX",
    "eclipsPM25":         "ECLIPSE_PM25",
    "lungCRt17":          "LUNG_CANCER_RATE",
    "physRt17":           "PHYSICAL_HEALTH_RATE",
    "adAsthma17":         "ADULT_ASTHMA_RATE",
    "cancerRt17":         "CANCER_RATE",
    "hyptRt17":           "HYPERTENSION_RATE",
}

df = df.rename(columns=rename_map)
print("Columns after rename:")
print(df.columns.tolist())

Columns after rename:
['GEOID', 'CEJI_SCORE', 'TREE_CANOPY_CHANGE', 'TREE_CANOPY_2017', 'TREE_CANOPY_2010', 'HEALTH_DEATH_PERCENT', 'LEAD_POISON_RATE', 'NDVI', 'TREE_COUNT', 'TREE_CROWN_DENSITY', 'SOCIAL_VULNERABILITY_INDEX', 'HARDSHIP INDEX', 'URBAN_FLOOD_RISK', 'SURFACE_TEMP', 'TRAFFIC_VOLUME', 'PM25', 'CHILD_ASTHMA_ED_RATE', 'CHILD_ASTHMA_RATE', 'Population', 'White', 'Black', 'NATIVE_AMERICAN', 'Asian', 'Hispanic', 'OTHER_RACE', 'CHILDREN_PERCENT', 'SENIOR_PERCENT', 'ZIP_CODE', 'COMMUNITY_AREA_NUMBER', 'Community Area', 'PLANT_DIVERSITY', 'PLANT_TOTAL', 'PLANT_SPECIES', 'RED_LINED', 'MEDIAN_RENT', 'MEDIAN_HOME_VALUE', 'FOREIGN_BORN', 'HOUSEHOLD_SIZE', 'FAMILY_HOUSEHOLD_PERCENT', 'BACHELORS_DEGREE', 'POPULATION_DENSITY', 'PERCENT OF HOUSING CROWDED', 'COST_BURDEN', 'HOUSING_SUSCEPTIBILITY', 'HOUSING_VULNERABILITY', 'HOUSING_CHANGE', 'DISPLACEMENT_PRESSURE', 'AIR_TEMP', 'HEAT_AVE', 'HEAT_MAX', 'ECLIPSE_PM25', 'LUNG_CANCER_RATE', 'PHYSICAL_HEALTH_RATE', 'ADULT_ASTHMA_RATE', 'CANCER_RA

In [4]:
# Add Latitude / Longitude from community area centroids
com_area = pd.to_numeric(df["COMMUNITY_AREA_NUMBER"], errors="coerce")
df["Latitude"]  = com_area.map(lambda x: CENTROIDS.get(int(x), (None, None))[0] if pd.notna(x) else None)
df["Longitude"] = com_area.map(lambda x: CENTROIDS.get(int(x), (None, None))[1] if pd.notna(x) else None)

missing = df["Latitude"].isna().sum()
print(f"Rows without lat/lon: {missing} / {len(df)}")
df[["Community Area", "COMMUNITY_AREA_NUMBER", "Latitude", "Longitude"]].head(10)

Rows without lat/lon: 0 / 801


,Community Area,COMMUNITY_AREA_NUMBER,Latitude,Longitude
0,NORTH LAWNDALE,29,41.8640,-87.7215
1,BRIGHTON PARK,58,41.8247,-87.7052
2,MCKINLEY PARK,59,41.8315,-87.6924
3,MCKINLEY PARK,59,41.8315,-87.6924
4,NEW CITY,61,41.8132,-87.6544
5,NEW CITY,61,41.8132,-87.6544
6,WASHINGTON PARK,40,41.8060,-87.6234
7,HYDE PARK,41,41.7985,-87.5913
8,WOODLAWN,42,41.7809,-87.5966
9,WOODLAWN,42,41.7809,-87.5966


In [5]:
# Preview domain-matched columns
domain_cols = [
    "Community Area", "HARDSHIP INDEX", "Population",
    "White", "Black", "Hispanic", "Asian",
    "MEDIAN_RENT", "MEDIAN_HOME_VALUE", "FOREIGN_BORN",
    "PERCENT OF HOUSING CROWDED", "BACHELORS_DEGREE",
    "Latitude", "Longitude",
]
df[domain_cols].head()

,Community Area,HARDSHIP INDEX,Population,White,Black,Hispanic,Asian,MEDIAN_RENT,MEDIAN_HOME_VALUE,FOREIGN_BORN,PERCENT OF HOUSING CROWDED,BACHELORS_DEGREE,Latitude,Longitude
0,NORTH LAWNDALE,21.3,NaN,64.4,12.6,73.8,1.0,944.0,193100.0,34.938525,2.333333,23.105360,41.8640,-87.7215
1,BRIGHTON PARK,46.6,NaN,45.5,1.5,84.0,9.1,921.0,162600.0,45.771670,15.292840,12.625538,41.8247,-87.7052
2,MCKINLEY PARK,27.7,NaN,36.4,1.6,48.2,33.2,938.0,269200.0,45.602606,2.694136,16.273187,41.8315,-87.6924
3,MCKINLEY PARK,37.4,NaN,47.7,2.1,69.0,15.7,907.0,221500.0,38.491011,10.805690,22.598870,41.8315,-87.6924
4,NEW CITY,40.0,NaN,50.6,1.2,97.5,0.0,762.0,128100.0,41.998990,8.233276,16.472868,41.8132,-87.6544


In [6]:
# Export
out_path = r"C:\Users\shanm\Downloads\chives-data-dashboard.csv"
df.to_csv(out_path, index=False)
print(f"Saved → {out_path}")
print(f"{len(df):,} rows × {len(df.columns)} columns")

Saved → C:\Users\shanm\Downloads\chives-data-dashboard.csv
801 rows × 58 columns
